# Create DeviceAssociation Table

Creates a managed Delta table over `Basic` for device-association records only.

The `Basic` table contains multiple FHIR resource types. Fabric IQ Ontology binds to a single managed table per entity type, so this notebook creates a filtered table with only the device-association records needed by the **DeviceAssociation** entity type in the `ClinicalDeviceOntology`.

### Prerequisites
- Attach this notebook to the **Silver Lakehouse** (the one containing the HDS `Basic` table)
- HDS clinical pipeline must have completed (Silver Lakehouse populated)

### Re-run
Re-run this notebook any time the `Basic` table is refreshed with new device associations.

In [4]:
CREATE OR REPLACE TABLE DeviceAssociation AS
SELECT
    id,
    idOrig,
    get_json_object(extension, '$[0].valueReference.reference') AS device_ref,
    get_json_object(subject_string, '$.display')                AS patient_name,
    get_json_object(subject_string, '$.idOrig')                 AS patient_id,
    get_json_object(code_string, '$.coding[0].code')            AS assoc_code,
    get_json_object(code_string, '$.coding[0].display')         AS assoc_display
FROM Basic
WHERE get_json_object(code_string, '$.coding[0].code') = 'device-assoc'

StatementMeta(, 287dc4ac-1abc-4ccb-87b7-d1fe403b57d9, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### Verify
Confirm the table was created with the expected row count (~100 rows, one per Masimo device).

In [5]:
SELECT COUNT(*) AS device_association_count FROM DeviceAssociation

StatementMeta(, 287dc4ac-1abc-4ccb-87b7-d1fe403b57d9, 7, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
SELECT * FROM DeviceAssociation LIMIT 10

StatementMeta(, 287dc4ac-1abc-4ccb-87b7-d1fe403b57d9, 8, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 7 fields>